# Notebook 10: Benchmark of next-visit diagnosis models, with a focus on rare labels

Every model below is trained and evaluated under one protocol (same cohort, splits, labels, optimiser,
early stopping, metrics), so that differences can be attributed to the model.

| family | models |
|---|---|
| baselines | `label_prior`, `bag_of_codes` |
| recurrent | `gru`, `retain` |
| Transformer (MedCo backbone and variants) | `transformer`, `transformer_pos_last`, `transformer_setpool`, `transformer_cross`, `transformer_logitadj` |
| knowledge | `gram`, `knowledge_rag_readout` |
| deep (non-looped) | `deep_L2`, `deep_L4` |
| looped | `loop_T4` (one layer x4), `layerloop_L2T2` (L1 L1 L2 L2), `modelloop_L2T2` (L1 L2 L1 L2) |
| adaptive depth | `deep_L4_adaptive`, `loop_T4_adaptive`, `loop_T4_gate`, `layerloop_L2T2_adaptive`, `modelloop_L2T2_adaptive` |

`layerloop_L2T2`, `modelloop_L2T2`, `loop_T4` and `deep_L4` all apply four Transformer blocks per modality
(same compute); the looped ones have the parameters of one or two blocks. Adaptive models are read out after
every block and trained with a truncated geometric prior over depths (or a PonderNet gate); at test time each
sample exits at the first depth whose top-$m$ entropy (or change since the previous depth) falls below a
threshold chosen on validation so as to keep 99% of the full-depth sample-AUPRC (Popescu et al., 2026).

**Labels.** The main benchmark uses the corrected ICD-10 $\rightarrow$ ICD-9 mapping (Notebook 9,
`scripts/relabel_gem.py`); `transformer` is also run on the original labels to connect with Notebook 2.

## How to run

*Parallel (recommended).* One job per model, at most four at a time (the account limit), from
`~/cambridge_project`:

```bash
python scripts/relabel_gem.py --processed processed --hosp data/hosp --out processed_gemfix   # once, on the login node
mkdir -p logs && sbatch scripts/benchmark.sbatch                                            # 21 models, seed 42
SEED=123 sbatch scripts/benchmark.sbatch; SEED=7 sbatch scripts/benchmark.sbatch            # more seeds
MODELS="transformer" PROCESSED=processed OUT=results/benchmark/original sbatch --array=0-0 scripts/benchmark.sbatch
```
then run this notebook with `BENCH_TRAIN=0` to aggregate.

*Sequential.* Running the notebook with `BENCH_TRAIN=1` trains every missing model in turn (several days on one
GPU slice; a run interrupted by the time limit resumes from its checkpoint). `BENCH_MODELS` selects a subset.

In [ ]:
import os, sys, json
from pathlib import Path
ROOT = Path(os.environ.get('RAREPRED_ROOT', '..')).resolve()
sys.path.insert(0, str(ROOT / 'src'))
PROCESSED   = Path(os.environ.get('RAREPRED_PROCESSED', ROOT / 'processed'))
EXTERNAL    = ROOT / 'data' / 'external'
RESULTS     = ROOT / 'results'
FIGURES     = ROOT / 'figures'
CHECKPOINTS = ROOT / 'checkpoints'
for d in (RESULTS, FIGURES, CHECKPOINTS):
    d.mkdir(parents=True, exist_ok=True)

import subprocess
import numpy as np, pandas as pd, torch
from rarepred.bench import MODELS, run
from rarepred.aggregate import load_runs, summarise, latex_main, latex_depth
PROC_FIX    = Path(os.environ.get('BENCH_PROCESSED', ROOT / 'processed_gemfix'))
PROC_ORIG   = PROCESSED
OUT_FIX     = RESULTS / 'benchmark' / 'gemfix'
OUT_ORIG    = RESULTS / 'benchmark' / 'original'
CKPT        = CHECKPOINTS / 'benchmark'
BENCH_TRAIN = os.environ.get('BENCH_TRAIN', '1') == '1'
SEEDS       = [int(s) for s in os.environ.get('BENCH_SEEDS', '42').split(',')]
SELECT      = [m for m in os.environ.get('BENCH_MODELS', '').split(',') if m] or list(MODELS)
MAX_EPOCHS  = int(os.environ.get('BENCH_MAX_EPOCHS', 50))
MAX_TRAIN   = int(os.environ['BENCH_MAX_TRAIN']) if os.environ.get('BENCH_MAX_TRAIN') else None
KNOWLEDGE   = PROC_FIX / f"knowledge_embeddings_{os.environ.get('RAREPRED_ENCODER_TAG', 'BioLORD-2023')}.npz"
TABLES      = ROOT / 'report' / 'tables'
TABLES.mkdir(parents=True, exist_ok=True)
print('device', 'cuda' if torch.cuda.is_available() else 'cpu', '| train', BENCH_TRAIN, '| seeds', SEEDS,
      '| models', len(SELECT))

import matplotlib.pyplot as plt
plt.rcParams.update({
    'figure.dpi': 110, 'savefig.dpi': 200, 'font.size': 9, 'axes.titlesize': 10,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.edgecolor': '#666666', 'axes.labelcolor': '#191919', 'xtick.color': '#666666',
    'ytick.color': '#666666', 'axes.grid': True, 'grid.color': '#e6e6e6', 'grid.linewidth': 0.6,
    'axes.axisbelow': True, 'legend.frameon': False,
})
BLUE, ORANGE, AQUA, GREY = '#257de0', '#f47230', '#00b894', '#9c9c9c'

## 1. Data with corrected labels

In [ ]:
if not (PROC_FIX / 'meta_ccs.json').exists():
    print(subprocess.run([sys.executable, str(ROOT / 'scripts' / 'relabel_gem.py'), '--processed', str(PROC_ORIG),
                          '--out', str(PROC_FIX)], capture_output=True, text=True).stdout)
meta_fix = json.load(open(PROC_FIX / 'meta_ccs.json'))
print(f"labels: {meta_fix['N']} ({meta_fix.get('label_space')}), rare stratum <= {meta_fix['q25']:.0f} training patients")

## 2. Knowledge embeddings (for `knowledge_rag_readout`)

Built by Notebook 5 on the corrected data:
`RAREPRED_PROCESSED=processed_gemfix NB=05_knowledge_base_rag sbatch ../scripts/run_notebook.sbatch`.
The model is skipped if the file is missing.

In [ ]:
print('knowledge embeddings:', KNOWLEDGE.name, 'found' if KNOWLEDGE.exists() else 'MISSING (knowledge_rag_readout skipped)')

## 3. Train and evaluate

In [ ]:
def done(out_dir, name, seed):
    return (Path(out_dir) / f"{name}_s{seed}.json").exists()

if BENCH_TRAIN:
    for seed in SEEDS:
        for name in SELECT:
            if name == 'knowledge_rag_readout' and not KNOWLEDGE.exists():
                continue
            if done(OUT_FIX, name, seed):
                print(f"{name} s{seed}: done")
                continue
            r = run(name, PROC_FIX, OUT_FIX, CKPT / 'gemfix', seed=seed, max_epochs=MAX_EPOCHS, max_train=MAX_TRAIN,
                    knowledge_npz=KNOWLEDGE)
            t = r['test']
            print(f"==> {name} s{seed}: sample-AUPRC {t['sample_auprc']:.4f} | Q1 {t['sample_auprc[Q1 (rarest)]']:.4f}")
        if (PROC_ORIG / 'meta_ccs.json').exists() and not done(OUT_ORIG, 'transformer', seed):
            run('transformer', PROC_ORIG, OUT_ORIG, CKPT / 'original', seed=seed, max_epochs=MAX_EPOCHS, max_train=MAX_TRAIN)

## 4. Results

In [ ]:
runs = load_runs(OUT_FIX)
print(f"{len(runs)} runs, {runs.model.nunique()} models, seeds {sorted(runs.seed.unique())}")
summary = summarise(runs)
summary.to_csv(RESULTS / 'benchmark_summary.csv')
latex_main(summary, TABLES / 'benchmark_main.tex')
latex_depth(summary, TABLES / 'benchmark_depth.tex')
cols = ['sample_auprc', 'sample_auprc[Q1 (rarest)]', 'sample_auprc[Q4 (common)]', 'gap_sample_auprc',
        'label_auprc[Q1 (rarest)]', 'medco_Q1', 'params', 'train_h']
view = summary[[f"{c}:mean" for c in cols] + ['n_seeds']]
view.columns = ['all', 'Q1', 'Q4', 'gap', 'label Q1', 'MedCo-groups Q1', 'params', 'train h', 'seeds']
view.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(6.2, 4.2))
fam_col = {'baseline': '#9c9c9c', 'recurrent': '#f2a600', 'transformer': '#257de0', 'knowledge': '#00b894',
           'deep': '#6a4ce2', 'looped': '#f47230', 'adaptive': '#e24d8b'}
for name, r in summary.iterrows():
    f = MODELS[name][0]
    ax.scatter(r['sample_auprc:mean'], r['sample_auprc[Q1 (rarest)]:mean'], s=28, color=fam_col[f], zorder=3)
    ax.annotate(name, (r['sample_auprc:mean'], r['sample_auprc[Q1 (rarest)]:mean']), fontsize=6,
                xytext=(3, 2), textcoords='offset points', color='#444444')
for f, c in fam_col.items():
    ax.scatter([], [], color=c, label=f, s=20)
ax.set_xlabel('test sample-AUPRC (all labels)'); ax.set_ylabel('test sample-AUPRC (rarest quartile)')
ax.legend(fontsize=7, loc='lower right'); ax.set_title('Overall vs rare-label performance', loc='left')
plt.tight_layout(); plt.savefig(FIGURES / 'benchmark_overall_vs_q1.pdf', bbox_inches='tight')
plt.savefig(ROOT / 'report' / 'figures' / 'benchmark_overall_vs_q1.pdf', bbox_inches='tight'); plt.show()

### Depth: deep vs looped, layer loop vs model loop, adaptive exit

In [ ]:
depth_models = [n for n in summary.index if any(f"depth{t}:sample_auprc:mean" in summary.columns and
                np.isfinite(summary.loc[n].get(f"depth{t}:sample_auprc:mean", np.nan)) for t in (1, 2, 3, 4))]
if depth_models:
    fig, axes = plt.subplots(1, 2, figsize=(8, 3.2), sharex=True)
    for i, n in enumerate(depth_models):
        ys = [summary.loc[n].get(f"depth{t}:sample_auprc:mean", np.nan) for t in (1, 2, 3, 4)]
        yq = [summary.loc[n].get(f"depth{t}:Q1:mean", np.nan) for t in (1, 2, 3, 4)]
        c = ['#257de0', '#f47230', '#00b894', '#6a4ce2', '#f2a600'][i % 5]
        axes[0].plot([1, 2, 3, 4], ys, marker='o', ms=3, lw=1.8, color=c, label=n)
        axes[1].plot([1, 2, 3, 4], yq, marker='o', ms=3, lw=1.8, color=c)
    axes[0].set_title('all labels', loc='left'); axes[1].set_title('rarest quartile', loc='left')
    for a in axes:
        a.set_xlabel('block applications (exit depth)'); a.set_xticks([1, 2, 3, 4])
    axes[0].set_ylabel('test sample-AUPRC'); axes[0].legend(fontsize=7)
    plt.tight_layout(); plt.savefig(ROOT / 'report' / 'figures' / 'benchmark_depth_curves.pdf', bbox_inches='tight'); plt.show()
ad_cols = [c for c in summary.columns if c.startswith('entropy@m:') or c.startswith('kl_consecutive:')]
summary.loc[[n for n in summary.index if MODELS[n][0] == 'adaptive'], [c for c in ad_cols if c.endswith(':mean')]].round(3)

### Original vs corrected labels (MedCo backbone)

In [ ]:
if OUT_ORIG.exists() and any(OUT_ORIG.glob('*.json')):
    o = summarise(load_runs(OUT_ORIG)).loc['transformer']
    f = summary.loc['transformer'] if 'transformer' in summary.index else None
    cmp_ = pd.DataFrame({'original labels': [o[f"{k}:mean"] for k, _ in [('sample_auprc', 0), ('sample_auprc[Q1 (rarest)]', 0), ('sample_auprc[Q4 (common)]', 0)]],
                         'corrected labels': [f[f"{k}:mean"] if f is not None else np.nan for k in ['sample_auprc', 'sample_auprc[Q1 (rarest)]', 'sample_auprc[Q4 (common)]']]},
                        index=['all', 'Q1', 'Q4'])
    display(cmp_.round(4))
    json.dump(cmp_.to_dict(), open(RESULTS / 'benchmark_original_vs_corrected.json', 'w'), indent=2)
else:
    print('no run on the original labels yet')

In [ ]:
print('tables written:', sorted(p.name for p in TABLES.glob('*.tex')))
print('recompile the report: cd report && latexmk -pdf main.tex')